In [ ]:
import pandas as pd
df = pd.read_csv("SpotifyFeatures.csv")
df.head()

Виконується імпорт бібліотеки pandas та завантаження датасету SpotifyFeatures.csv у DataFrame.
Після завантаження виводяться перші рядки таблиці для ознайомлення зі структурою даних.

In [ ]:
from IPython.display import display
print("HEAD:")
display(df.head())

print("\nSHAPE:")
print(df.shape)

print("\nDTYPES:")
display(df.dtypes)

print("\nMISSING VALUES:")
display(df.isnull().sum())

print("\nDESCRIBE:")
display(df.describe())

Виконується базовий аналіз датасету.
виводяться перші рядки таблиці (head), щоб оцінити структуру даних;
визначається розмірність таблиці (shape);
перевіряються типи даних (dtypes);
аналізується наявність пропущених значень (isnull);
обчислюється описова статистика (describe) для числових ознак.

In [ ]:
features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'duration_ms',
    'popularity'
]
X = df[features]
X.head()

Обираються числові ознаки, які будуть використані для кластеризації. Для цього було сформовано нову таблицю X, яка містить лише вибрані числові ознаки для подальшої обробки.

In [ ]:
X.isnull().sum()

Перед застосуванням алгоритму кластеризації необхідно перевірити наявність пропущених значень у вибраних ознаках.
Алгоритм K-Means не працює з пропущеними значеннями, тому у випадку їх наявності потрібно виконати обробку.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:5]

Перед застосуванням алгоритму K-Means необхідно виконати стандартизацію ознак.
Це пов’язано з тим, що різні ознаки мають різний масштаб, що може спотворювати результати кластеризації.
Для цього використовується StandardScaler, який перетворює дані таким чином, що кожна ознака має середнє значення 0 та стандартне відхилення 1.

In [ ]:
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
X_scaled_df.head()

Після стандартизації дані перетворюються у вигляд масиву, який містить нормалізовані значення ознак.

In [ ]:
df_popular = df[df['popularity'] >= 85]
df_popular.head()

Для зменшення кількості точок на графіках залишимо для візуалізації лише популярні треки, у яких значення popularity не менше 85.


In [ ]:
df_popular.shape

Розмірність таблиці після фільтрації.

In [ ]:
from sklearn.cluster import KMeans
k_range = range(2, 16)
inertia = []

На цьому етапі визначається оптимальна кількість кластерів для алгоритму K-Means.
Для цього використовується метод ліктя, який базується на аналізі інерції — суми квадратів відстаней від кожної точки до центру її кластера.
Для визначення оптимальної кількості кластерів запускаємо алгоритм K-Means для різних значень k у діапазоні від 2 до 15.

In [ ]:
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

Для кожного значення k обчислюється інерція — сума квадратів відстаней від кожної точки до найближчого центру кластера.
Менше значення інерції означає кращу компактність кластерів.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.plot(k_range, inertia, marker='o')
plt.xlabel("Кількість кластерів (k)")
plt.ylabel("Інерція")
plt.title("Метод ліктя")
plt.grid()
plt.show()

Будується графік залежності інерції від кількості кластерів k.
Зі збільшенням k інерція зменшується, але в певний момент зменшення стає менш значним — ця точка і є "лікоть".

На основі побудованого графіка визначаємо лікоть, де швидкість зменшення інерції різко змінюється.
Це значення k = 4 і вважається оптимальним, оскільки подальше збільшення кількості кластерів не дає суттєвого покращення якості кластеризації. Як ми визначили: спостерігається різке зменшення інерції при збільшенні кількості кластерів від 2 до 4, а після значення k = 4 зменшення інерції відбувається значно повільніше, що і свідчить про правильне значення

In [ ]:
k_opt = 4
kmeans = KMeans(n_clusters=k_opt, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

Після визначення оптимальної кількості кластерів методом ліктя виконується безпосередня кластеризація даних за допомогою алгоритму K-Means.
У попередньому пункті було обрано значення k = 4, тому всі треки будуть поділені на 4 кластери.
На цьому етапі створюється модель K-Means з кількістю кластерів k = 4.

In [ ]:
df['cluster'] = clusters
df[['track_name', 'artist_name', 'cluster']].head()

Після виконання кластеризації кожному треку присвоюється номер кластера. Отримані мітки додаються до початкової таблиці як нова колонка cluster.

In [ ]:
cluster_counts = df['cluster'].value_counts().sort_index()
cluster_counts

Підрахуємо, скільки треків потрапило до кожного кластера. Це дозволяє оцінити, чи є кластери приблизно збалансованими за кількістю об'єктів.

In [ ]:
plt.figure(figsize=(7, 5))
cluster_counts.plot(kind='bar')
plt.xlabel("Кластер")
plt.ylabel("Кількість треків")
plt.title("Кількість треків у кожному кластері")
plt.grid(axis='y')
plt.show()

Побудуємо стовпчиковий графік, який показує кількість треків у кожному кластері.
Спостерігається нерівномірний розподіл об'єктів. Це свідчить про те, що дані мають нерівномірну структуру, і деякі типи треків зустрічаються значно частіше за інші. Зокрема, кластер 1 є найменшим, що може вказувати на специфічний вид треків.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

Для візуалізації результатів кластеризації використовується метод головних компонент (PCA), який дозволяє зменшити розмірність даних до 2 або 3 вимірів. Це дає можливість відобразити багатовимірні дані на площині та побачити структуру кластерів.

Метод PCA (Principal Component Analysis) дозволяє нам перетворити початкові ознаки у новий простір головних компонент, які зберігають найбільшу частину інформації (дисперсії). Спочатку для візуалізації використаємо 2 компоненти.

In [ ]:
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]
df[['PC1', 'PC2', 'cluster']].head()

Додамо отримані головні компоненти до DataFrame для подальшої візуалізації.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=df,
    x='PC1',
    y='PC2',
    hue='cluster',
    palette='viridis'
)
plt.title("Візуалізація кластерів (PCA)")
plt.xlabel("Головна компонента 1")
plt.ylabel("Головна компонента 2")
plt.legend(title="Кластер")
plt.grid()
plt.show()

Кожна точка відповідає одному треку, а колір визначає належність до певного кластера.

На графіку візуалізації кластерів у просторі головних компонент чітко спостерігається поділ даних на чотири групи.
Кластери розташовані компактно та мають відносно чіткі межі, особливо вздовж першої головної компоненти. Це свідчить про те, що алгоритм K-Means успішно виділив структуру в даних. Деяке перекривання кластерів спостерігається, що є типовим для реальних даних та пояснюється зменшенням розмірності за допомогою PCA.

Обране значення k = 4 є обґрунтованим і добре відображає структуру датасету.

In [ ]:
from sklearn.decomposition import PCA

pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

df['PC1_3D'] = X_pca_3d[:, 0]
df['PC2_3D'] = X_pca_3d[:, 1]
df['PC3_3D'] = X_pca_3d[:, 2]

Для більш детального аналізу структури даних виконаємо зменшення розмірності до трьох головних компонент та побудуємо тривимірний графік.
Це дозволяє краще побачити розділення кластерів у просторі ознак.

In [ ]:
import plotly.express as px
fig = px.scatter_3d(
    df,
    x='PC1_3D',
    y='PC2_3D',
    z='PC3_3D',
    color='cluster',
    color_continuous_scale='viridis',
    title="3D візуалізація кластерів (PCA)"
)
fig.show()

Тривимірна візуалізація кластерів дозволяє більш чітко побачити їх структуру та відокремленість у просторі ознак.
у 3D краще помітно розділення між кластерами, що підтверджує коректність обраного значення k = 4.

In [ ]:
cluster_stats = df.groupby('cluster')[features].mean()
cluster_stats

На цьому етапі виконується аналіз отриманих кластерів. Для кожного кластера обчислюються середні значення ознак, що дозволяє визначити характерні особливості кожної групи треків.

In [ ]:
X_scaled_df['cluster'] = df['cluster']
cluster_stats_scaled = X_scaled_df.groupby('cluster').mean()
cluster_stats_scaled

Для коректної візуалізації порівняння ознак між кластерами використаємо стандартизовані значення, щоб уникнути впливу різних масштабів.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

sns.heatmap(
    cluster_stats_scaled,
    annot=True,
    cmap='coolwarm',
    center=0
)

plt.title("Характеристики кластерів")
plt.xlabel("Ознаки")
plt.ylabel("Кластери")

plt.show()

Для наочного порівняння кластерів будується heatmap, яка відображає середні значення стандартизованих ознак у кожному кластері.
Найбільш відрізняльними ознаками є speechiness, energy та acousticness.

Інтерпретація кластерів

На основі середніх значень ознак  було виконано інтерпретацію кожного кластера.

Кластер 0

Характеризується високими значеннями energy (0.71) та loudness (0.58), що свідчить про енергійні та гучні треки. Також спостерігається підвищений tempo (0.82), що вказує на швидкий ритм композицій. Danceability знаходиться на середньому рівні, а acousticness та instrumentalness низькі, що означає переважно електронні або сучасні композиції з вокалом.
Інтерпретація: енергійні, швидкі, танцювальні треки.


Кластер 1

Має дуже високе значення speechiness (4), що є ключовою ознакою цього кластера. Також спостерігається високе acousticness (1.2) та liveness (2.8). Також має низький tempo та popularity. popularity найнижча серед усіх кластерів.

Інтерпретація: розмовні треки, подкасти або живі записи.


Кластер 2

Характеризується найвищим значенням danceability (0.7) та valence (0.45), що вказує на позитивний настрій композицій. Energy середня (0.13), tempo помірний, а popularity найвища серед кластерів (0.35).

Інтерпретація: популярні, легкі, танцювальні та позитивні треки.


Кластер 3

Має дуже низькі значення energy (-1.4) та loudness (найбільш негативне значення), що означає тихі композиції. Водночас дуже високі значення acousticness (1.3) та instrumentalness (1.1) вказують на акустичні та інструментальні треки. Tempo низький, valence також нижчий.

Інтерпретація: спокійні, акустичні або інструментальні композиції.

Загальний висновок

Отримані кластери мають чітку інтерпретацію та відображають різні типи музичного контенту:
- енергійні танцювальні треки
- розмовні або live записи
- популярні позитивні композиції
- спокійні акустичні треки

Це підтверджує ефективність застосування методу K-Means для кластеризації музичних даних.

ВИСНОВОК


У ході роботи було виконано кластеризацію музичних треків за допомогою алгоритму K-Means. Дані були попередньо оброблені та стандартизовані, після чого методом ліктя визначено оптимальну кількість кластерів (k = 4). Для візуалізації результатів застосовано метод головних компонент (PCA), що дозволило відобразити структуру даних у 2D та 3D просторі. Проведений аналіз показав, що кластери мають змістовну інтерпретацію та відповідають різним типам музичного контенту. Отримані результати підтверджують ефективність використання методів кластеризації для аналізу музичних даних.